<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c40-header.png" alt="Nextia Learning · Experiments and Causal Impact: Did Our Change Work?" width="100%">

# Solution: The result and its uncertainty

**[The result and its uncertainty](https://learning.nextia-ai.com/courses/experiments/m03/the-result-and-its-uncertainty/)** · Experiments and Causal Impact: Did Our Change Work? · Module 3, lesson 1 of 4 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** report a test's effect with its range: the difference of means, Welch's t-test, a bootstrap interval, and a difference of proportions, each checked against the true effect.

| | |
|---|---|
| **Time** | About 40 minutes |
| **Needs** | An internet connection for the setup cells. No account and no key. |
| **You should know** | The test plan, from [Write the plan first](https://learning.nextia-ai.com/courses/experiments/m02/write-the-plan-first/); confidence intervals, from the [mathematics course](https://learning.nextia-ai.com/courses/math/). |
| **Tested** | Python 3.12 with pandas 3.1.1, SciPy 1.15.3, statsmodels 0.15.0 and matplotlib 3.11.2, in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/experiments/m03/the-result-and-its-uncertainty/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C40/M03-L01-the-result-and-its-uncertainty/the-result-and-its-uncertainty-solution.ipynb).

## Setup: install the packages

Run the next cell first. It installs the versions that the course uses: pandas 3.1.1, SciPy 1.15.3 and statsmodels 0.15.0, and matplotlib 3.11.2 for the charts. It needs an internet connection and takes about a minute the first time.

You should see one line per package, with these versions. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. Restarting is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==3.1.1 "scipy==1.15.3; python_version < '3.14'" "scipy==1.18.1; python_version >= '3.14'" statsmodels==0.15.0 matplotlib==3.11.2
import importlib.metadata as _md
for _name in ["pandas", "scipy", "statsmodels", "matplotlib"]:
    print(_name, _md.version(_name))

## Setup: get the practice data

Run the next cell. It downloads the simulator's data for this lesson (less than 3 MB) into a folder `larkfield-experiments/data`, and checks each file's checksum. It needs an internet connection and no account.

You should see `Ready: 2 files in data/.` If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-experiments` (in Colab: the **Files** panel on the left), then run the cell again.

The Larkfield data is synthetic: the course simulator made it. It has no real people. Each folder has `truth.json`: the true effects that the simulator knows. A real test never has this file.

In [ ]:
# Setup: download the course data into larkfield-experiments/data/ and check it.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

DATA = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C40/data/"
FILES = {
    "delivery/customers.csv.gz": "c22521f272d36b36fd26234d53d04aac1c8be347a5014ee3e2aeeffcc34a0c19",
    "delivery/truth.json": "6f088f29a112de123bb1da3bac0940a28c5adb706e9f24007686c373c14ab6a4"
}
LOCAL = os.environ.get("NX_DATA_LOCAL")  # authors only: a local copy of the labs data
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
PROJECT = Path("larkfield-experiments").resolve()
PROJECT.mkdir(exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / "data" / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists() and sha256(path) == expected:
        continue
    if LOCAL:
        shutil.copy(Path(LOCAL) / name, path)
    else:
        download(DATA + name, path)
    if sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-experiments and run again.")
os.chdir(PROJECT)  # work in the project folder, as in the lesson
print(f"Ready: {len(FILES)} files in data/.")

## Setup: helpers

The next cell loads pandas as `pd`, NumPy as `np`, SciPy's `stats` and statsmodels' formula interface as `smf`, and defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.
- `truth(name)` reads a scenario's `truth.json`: the simulator's true effects.

Run the cell. It prints nothing.

In [ ]:
# Helpers: checks that do not show the answer, and the simulator's truth.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy import stats
pd.set_option("display.precision", 4)
pd.set_option("display.width", 140)
warnings.filterwarnings("ignore")

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

def truth(name):
    """The simulator's true effects for a scenario (a real test never has this)."""
    return json.loads(open(f"data/{name}/truth.json").read())

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. Hana's test: free delivery from 50 dollars

For 28 days, a coin flip gave each customer free delivery from 75 dollars (**A**, today) or from 50 dollars (**B**). One row per customer who visited. The primary metric of the plan is **revenue per customer**.

> **Predict.** Most customers spend nothing in four weeks, and a few spend a lot. Will the mean or the median show the change better?

In [ ]:
d = pd.read_csv("data/delivery/customers.csv.gz")
print(d.groupby("variant")["revenue"].describe()[["count", "mean", "std", "50%", "max"]])
print("share with no order:", round((d["orders"] == 0).mean(), 3))

> **Check.** You should see a mean of 25.0504 (A) and 26.0770 (B) dollars, a median of 0, and `share with no order: 0.78`.

The median is 0 in both groups: it cannot see a change. The mean can, but it is noisy, because a few big orders move it.

## 2. Welch's t-test, step by step

In [ ]:
a, b = d.loc[d.variant == "A", "revenue"], d.loc[d.variant == "B", "revenue"]
diff = b.mean() - a.mean()
se = np.sqrt(a.var() / len(a) + b.var() / len(b))
low, high = diff - 1.96 * se, diff + 1.96 * se
t = stats.ttest_ind(b, a, equal_var=False)
print(f"difference {diff:+.3f} dollars ({diff / a.mean():+.1%}), SE {se:.3f}, 95% interval {low:+.3f} to {high:+.3f}, p = {t.pvalue:.4f}")

> **Check.** You should see `difference +1.027 dollars (+4.1%), SE 0.443, 95% interval +0.158 to +1.895, p = 0.0205`.

Report it as Hana will read it: "free delivery from 50 dollars raised revenue per customer by about 1 dollar (4%), and the effect is between 0.16 and 1.90 dollars." The p-value alone ("p = 0.0205") hides how big the effect is.

## 3. The bootstrap: the same interval without a formula

Resample each group with replacement many times, and look at the spread of the differences. It needs no formula, only a computer.

In [ ]:
rng = np.random.default_rng(0)
xa, xb = a.to_numpy(), b.to_numpy()
boot = np.array([rng.choice(xb, len(xb)).mean() - rng.choice(xa, len(xa)).mean() for _ in range(1000)])
print(f"bootstrap 95% interval {np.percentile(boot, 2.5):+.3f} to {np.percentile(boot, 97.5):+.3f}")

> **Check.** You should see `bootstrap 95% interval +0.193 to +1.859`, close to Welch's.

## 4. Check against the truth

In [ ]:
tr = truth("delivery")
print(f"true effect on revenue {tr['revenue']['effect']:+.3f} dollars ({tr['revenue']['relative']:+.1%}); inside Welch's interval: {low <= tr['revenue']['effect'] <= high}")

> **Check.** You should see `true effect on revenue +0.698 dollars (+2.8%); inside Welch's interval: True`.

Our estimate (+1.027) is higher than the truth (+0.698): this run was a little lucky. Over 200 seeds of this scenario, the course measured that Welch's interval contains the truth in 95.5% of them, and finds a clear effect in only 41.5%: the test is too small for an effect of this size (see [Sample size and power](https://learning.nextia-ai.com/courses/experiments/m02/sample-size-and-power/)).

## 5. Your turn: the share of customers who ordered

> **Your turn.** Compute the 95% interval of the difference (B − A) in the share of customers who ordered at least once (`orders > 0`), with the formula for two proportions: SE = √(pA(1 − pA)/nA + pB(1 − pB)/nB). Put the lower end in `conv_low`, rounded to 4 decimals.

In [ ]:
pa = (d.loc[d.variant == "A", "orders"] > 0).mean()
pb = (d.loc[d.variant == "B", "orders"] > 0).mean()
na, nb = len(a), len(b)
se_c = np.sqrt(pa * (1 - pa) / na + pb * (1 - pb) / nb)
conv_low = round(pb - pa - 1.96 * se_c, 4)
print(pa, pb, conv_low)

In [ ]:
expect('conv_low', conv_low, 0.0066)

## Next

You can report an effect with its range. The [lesson page](https://learning.nextia-ai.com/courses/experiments/m03/the-result-and-its-uncertainty/) has the diagrams and the knowledge checks that count toward your certificate. Next, we check the test itself before we trust its result.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Check the test before the result](https://learning.nextia-ai.com/courses/experiments/m03/check-the-test-before-the-result/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/experiments/m03/the-result-and-its-uncertainty/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.